##Project: Retail Store Sales Analysis

######Prepared by Ashwin Trivedi
######2026B15410116
######Github link- https://github.com/Ashwin-Trivedi/PYTHON_PROJECT/

In [ ]:
import pandas as pd

file_path = "/content/sample_data/retail.csv"

df = pd.read_csv(file_path)
df["Date"] = pd.to_datetime(df["Date"])

print("Loaded shape (rows, columns):", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nMissing values per column:")
print(df.isnull().sum())

Loaded shape (rows, columns): (27, 12)

First 5 rows:
   Transaction_ID       Date Customer_ID     Branch     Category     Product  \
0               1 2024-07-13    CUST-187     Mumbai      Grocery        Rice   
1               2 2024-08-21    CUST-218  Ghaziabad  Electronics      Mobile   
2               3 2024-10-16    CUST-114      Delhi     Clothing       Jeans   
3               4 2024-01-16    CUST-120      Noida  Electronics  Headphones   
4               5 2024-05-15    CUST-179  Bangalore         Home       Chair   

   Quantity  Unit_Price  Discount  Total_Amount    Month    Weekday  
0       4.0      4116.0       0.0      16464.00  2024-07   Saturday  
1       2.0      2832.0       0.0       5664.00  2024-08  Wednesday  
2       2.0      2938.0       5.0       5582.20  2024-10  Wednesday  
3       3.0      2186.0      30.0       4590.60  2024-01    Tuesday  
4       3.0       235.0       5.0        669.75  2024-05  Wednesday  

Missing values per column:
Transaction_ID   

In [ ]:
#Q1 — Overall Sales Value & Trend Over Time

total_sales = df["Total_Amount"].sum()
avg_txn = df["Total_Amount"].mean()
median_txn = df["Total_Amount"].median()

print(f"Total Sales: {total_sales:,.2f}")
print(f"Average Transaction Value: {avg_txn:,.2f}")
print(f"Median Transaction Value: {median_txn:,.2f}")

monthly_sales = (
    df.groupby("Month")
    .agg(
        Total_Sales=("Total_Amount", "sum"),
        Transactions=("Transaction_ID", "count"),
        Avg_Txn_Value=("Total_Amount", "mean"),
    )
    .sort_index()
)
monthly_sales["MoM_Pct_Change"] = monthly_sales["Total_Sales"].pct_change() * 100

print("\nMonthly Sales Summary:")
print(monthly_sales.round(2))

Total Sales: 180,971.05
Average Transaction Value: 6,960.43
Median Transaction Value: 5,086.40

Monthly Sales Summary:
         Total_Sales  Transactions  Avg_Txn_Value  MoM_Pct_Change
Month                                                            
2024-01     12342.60             2        6171.30             NaN
2024-03       268.20             1         268.20          -97.83
2024-04     40584.40             4       10146.10        15032.14
2024-05     29468.75             3        9822.92          -27.39
2024-06     16234.00             2        8117.00          -44.91
2024-07     25951.20             2       12975.60           59.86
2024-08     15559.20             3        5186.40          -40.04
2024-09      6046.20             3        2015.40          -61.14
2024-10      5582.20             1        5582.20           -7.67
2024-11     10809.50             2        5404.75           93.64
2024-12     18124.80             3        6041.60           67.67


In [ ]:
#Q2 — Categories & Products Revenue Share

category_rev = (
    df.groupby("Category")
    .agg(
        Revenue=("Total_Amount", "sum"),
        Share_Pct=("Total_Amount", lambda s: s.sum() / total_sales * 100),
        Transactions=("Transaction_ID", "count"),
        Total_Qty=("Quantity", "sum"),
    )
    .sort_values("Revenue", ascending=False)
)

product_rev = (
    df.groupby(["Category", "Product"])
    .agg(
        Revenue=("Total_Amount", "sum"),
        Share_Pct=("Total_Amount", lambda s: s.sum() / total_sales * 100),
        Transactions=("Transaction_ID", "count"),
        Total_Qty=("Quantity", "sum"),
    )
    .sort_values("Revenue", ascending=False)
)

print("Category Revenue Share:")
print(category_rev.round(2))

print("\nTop 5 Products by Revenue:")
print(product_rev.head(5).round(2))

print("\nBottom 5 Products by Revenue:")
print(product_rev.tail(5).round(2))

Category Revenue Share:
              Revenue  Share_Pct  Transactions  Total_Qty
Category                                                 
Clothing     61149.60      33.79             8       23.0
Grocery      50673.20      28.00             5       18.0
Electronics  43635.50      24.11             8       28.0
Home         25512.75      14.10             5       13.0

Top 5 Products by Revenue:
                        Revenue  Share_Pct  Transactions  Total_Qty
Category    Product                                                
Grocery     Rice        45375.2      25.07             3       11.0
Clothing    Jeans       33330.6      18.42             4       11.0
Electronics Mobile      19580.0      10.82             2        6.0
Clothing    Jacket      17784.0       9.83             3        7.0
Electronics Headphones  15419.6       8.52             3       10.0

Bottom 5 Products by Revenue:
                      Revenue  Share_Pct  Transactions  Total_Qty
Category    Product        

In [ ]:
# Q3 — Branch Performance

branch_perf = (
    df.groupby("Branch")
    .agg(
        Revenue=("Total_Amount", "sum"),
        Share_Pct=("Total_Amount", lambda s: s.sum() / total_sales * 100),
        Transactions=("Transaction_ID", "count"),
        Avg_Txn_Value=("Total_Amount", "mean"),
        Total_Qty=("Quantity", "sum"),
    )
    .sort_values("Revenue", ascending=False)
)

print("Branch Performance:")
print(branch_perf.round(2))

Branch Performance:
            Revenue  Share_Pct  Transactions  Avg_Txn_Value  Total_Qty
Branch                                                                
Noida      47328.00      26.15             6        7888.00       21.0
Delhi      38133.70      21.07             5        7626.74       17.0
Bangalore  36365.15      20.09             6        6060.86       16.0
Mumbai     34386.60      19.00             5        6877.32       17.0
Ghaziabad  24757.60      13.68             4        6189.40       11.0


In [ ]:
# Q4 — Months, Weekdays & Periods with Stronger/Weaker Sales

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

weekday_sales = (
    df.groupby("Weekday")
    .agg(
        Total_Sales=("Total_Amount", "sum"),
        Transactions=("Transaction_ID", "count"),
        Avg_Txn_Value=("Total_Amount", "mean"),
    )
    .reindex(weekday_order)
    .dropna(how="all")
)

df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)
quarter_sales = (
    df.groupby("Quarter")
    .agg(
        Total_Sales=("Total_Amount", "sum"),
        Transactions=("Transaction_ID", "count"),
        Avg_Txn_Value=("Total_Amount", "mean"),
    )
    .sort_index()
)

print("Sales by Weekday:")
print(weekday_sales.round(2))

print("\nSales by Quarter:")
print(quarter_sales.round(2))

Sales by Weekday:
           Total_Sales  Transactions  Avg_Txn_Value
Weekday                                            
Monday          545.40             1         545.40
Tuesday       43735.20             6        7289.20
Wednesday     11915.95             3        3971.98
Thursday      33468.90             6        5578.15
Friday        59129.80             6        9854.97
Saturday      16732.20             2        8366.10
Sunday        15443.60             2        7721.80

Sales by Quarter:
         Total_Sales  Transactions  Avg_Txn_Value
Quarter                                          
2024Q1      12610.80             3        4203.60
2024Q2      86287.15             9        9587.46
2024Q3      47556.60             8        5944.58
2024Q4      34516.50             6        5752.75
NaT             0.00             1            NaN


In [ ]:
# Q5 — Average Transaction Value Across Categories & Locations

atv_pivot = df.pivot_table(
    index="Category",
    columns="Branch",
    values="Total_Amount",
    aggfunc="mean",
)

print("Average Transaction Value (Category x Branch):")
print(atv_pivot.round(2))

Average Transaction Value (Category x Branch):
Branch       Bangalore     Delhi  Ghaziabad  Mumbai     Noida
Category                                                     
Clothing       7178.67   5582.20    6364.53     NaN  14937.80
Electronics        NaN   4375.83    5664.00  3137.4   7235.53
Grocery            NaN  19424.00        NaN  7812.3       NaN
Home           4943.05       NaN        NaN     NaN   5341.80


In [ ]:
# Q6 — Discounts vs. Quantities & Transaction Values

discount_summary = (
    df.groupby("Discount")
    .agg(
        Transactions=("Transaction_ID", "count"),
        Avg_Quantity=("Quantity", "mean"),
        Avg_Total_Amount=("Total_Amount", "mean"),
        Median_Total_Amount=("Total_Amount", "median"),
    )
    .sort_index()
)

print("Summary by Discount (%):")
print(discount_summary.round(2))
print("\nCorrelation (Discount vs Quantity):", round(df["Discount"].corr(df["Quantity"]), 4))
print("Correlation (Discount vs Total_Amount):", round(df["Discount"].corr(df["Total_Amount"]), 4))

Summary by Discount (%):
          Transactions  Avg_Quantity  Avg_Total_Amount  Median_Total_Amount
Discount                                                                   
0.0                  5          3.60          11892.80              13916.0
5.0                  5          3.20           5325.13               3117.9
10.0                 4          2.25           3268.80               1386.0
20.0                 6          3.67           9232.53               8619.6
30.0                 6          2.83           4401.83               3864.0

Correlation (Discount vs Quantity): -0.0901
Correlation (Discount vs Total_Amount): -0.2265


In [ ]:
# Q7 & Q8 — Frequent/High-Value Customers & Business Actions

customer_summary = df.groupby("Customer_ID").agg(
    Purchase_Count=("Transaction_ID", "count"),
    Total_Spent=("Total_Amount", "sum"),
    Avg_Order_Value=("Total_Amount", "mean"),
    Total_Qty=("Quantity", "sum"),
)

print("Top 5 Customers by Total Spend:")
print(customer_summary.sort_values("Total_Spent", ascending=False).head(5).round(2))

print("\nTop 5 Customers by Purchase Count:")
print(customer_summary.sort_values("Purchase_Count", ascending=False).head(5).round(2))

# Identify above-average Category x Branch pairs for non-discount promotion/stocking
high_atv_pairs = atv_pivot[atv_pivot > avg_txn].stack().reset_index()
high_atv_pairs.columns = ["Category", "Branch", "Avg_Txn_Value"]
print("\nHigh-ATV Category x Branch Pairs (Above Overall Mean):")
print(high_atv_pairs.sort_values("Avg_Txn_Value", ascending=False).round(2).to_string(index=False))

Top 5 Customers by Total Spend:
             Purchase_Count  Total_Spent  Avg_Order_Value  Total_Qty
Customer_ID                                                         
CUST-177                  2      32680.0          16340.0        9.0
CUST-5                    1      19424.0          19424.0        4.0
CUST-187                  1      16464.0          16464.0        4.0
CUST-292                  1      14937.8          14937.8        4.0
CUST-221                  1      13891.2          13891.2        4.0

Top 5 Customers by Purchase Count:
             Purchase_Count  Total_Spent  Avg_Order_Value  Total_Qty
Customer_ID                                                         
CUST-177                  2      32680.0          16340.0        9.0
CUST-167                  2       5232.4           2616.2        4.0
CUST-104                  1       1302.0           1302.0        3.0
CUST-113                  1       3117.9           3117.9        2.0
CUST-120                  1       4